In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
num_objects_list = range(int(1e5), int(9e5), int(1e5))

# Lists to store extracted data

def read_exp(dirname):
    num_objects = []
    tree_times = []
    force_times = []
    leap_times = []
    for n in num_objects_list:
        filepath = f'{dirname}/output_stat_{n}.txt'
        
        if os.path.exists(filepath):
            with open(filepath, 'r') as f:
                lines = f.readlines()
                # Store values for each file
                data = {}
                for line in lines:
                    if ':' in line:
                        parts = line.split(':')
                        label = parts[0].strip()
                        value = float(parts[1].strip())
                        data[label] = value
                    elif 'Num objects' in line:
                        # Capture the actual N from the file content if needed
                        current_n = int(line.split()[-1])
                
                num_objects.append(n)
                tree_times.append(data.get('time taken by tree construction and deletion'))
                force_times.append(data.get('time taken by force computation'))
                leap_times.append(data.get('time taken by leapfrog integration'))
    return np.array(num_objects), np.array(tree_times), np.array(force_times), np.array(leap_times)

In [ ]:
num_objects, tree_times, force_times, leap_times = read_exp('exp7')

# Plotting the results
plt.figure(figsize=(10, 6))
plt.plot(num_objects, tree_times, marker='o', label='Tree Construction/Deletion')
plt.plot(num_objects, force_times, marker='s', label='Force Computation')
plt.plot(num_objects, leap_times, marker='^', label='Leapfrog Integration')

plt.xlabel('Number of Objects ($N$)')
plt.ylabel('Time taken (seconds)')
plt.title('Run Times vs. Number of Objects')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

In [ ]:
total_times = tree_times + force_times + leap_times
tree_pct = (tree_times / total_times) * 100
force_pct = (force_times / total_times) * 100
leap_pct = (leap_times / total_times) * 100

labels = [f'{int(n/1000)}k' for n in num_objects]
width = 0.65

plt.figure(figsize=(12, 7))

# Plot bars (stacked)
p1 = plt.bar(labels, tree_pct, width, label='Tree Construction', color='#1f77b4')
p2 = plt.bar(labels, force_pct, width, bottom=tree_pct, label='Force Computation', color='#ff7f0e')
p3 = plt.bar(labels, leap_pct, width, bottom=tree_pct + force_pct, label='Leapfrog Integration', color='#2ca02c')

# Formatting
plt.ylabel('Percentage of Total Run Time (%)')
plt.xlabel('Number of Objects ($N$)')
plt.title('Percentage Breakdown of Run Time by Computation Step')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.ylim(0, 120)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Add text labels for percentages inside the bars
for i in range(len(labels)):
    plt.text(i, tree_pct[i]/2, f'{tree_pct[i]:.1f}%', ha='center', va='center', color='white', fontweight='bold')
    plt.text(i, tree_pct[i] + force_pct[i]/2, f'{force_pct[i]:.1f}%', ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()

In [ ]:


# 1. Extract data for each experiment
# Assuming read_exp returns numpy arrays as described
exps = {}
exp_indices = range(1, 8)

for i in exp_indices:
    dir_name = f'exp{i}'
    # Unpack the data returned by your function
    n_obj, t_tree, t_force, t_leap = read_exp(dir_name)
    
    exps[i] = {
        'n': n_obj,
        'tree': t_tree,
        'force': t_force,
        'leap': t_leap,
        'total': t_tree + t_force + t_leap
    }

# Define baseline (exp1)
baseline = exps[1]
n_axis = baseline['n']

# 2. Setup Plotting
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
#plt.subplots_adjust(wspace=0.1, bottom=0.5) # Increase bottom margin for legend
plt.subplots_adjust(wspace=0.1, bottom=0.5, top=0.8)
metrics = [
    ('total', 'Total Run Time Speedup', 0),
    ('force', 'Force Computation Speedup', 1),
    ('tree', 'Tree Computation Speedup', 2)
]

optim_names=['baseline','postCOM', 'Node_Contig', 'iterative', 'Body reorder', 'Body blocking', 'AVX']

# 3. Calculate and Plot Speedup for Exp2 and Exp3
for i in range(2,8):
    for key, title, ax_idx in metrics:
        # Speedup = Baseline_Time / Optimized_Time
        speedup = baseline[key] / exps[i][key]

        
        print(optim_names[i-1], key ,speedup[-1])
        
        axes[ax_idx].plot(n_axis, speedup, marker='o', label=optim_names[i-1])
        axes[ax_idx].set_title(title)
        axes[ax_idx].set_ylabel('Speedup Ratio ($x$)')
        axes[ax_idx].set_xlabel('Number of Objects')
        axes[ax_idx].grid(True, linestyle='--', alpha=0.6)
    
        # Add a reference line at y=1 (No speedup)
        axes[ax_idx].axhline(1, color='red', linestyle=':', alpha=0.5, label="_nolegend_")

# Add legends

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.02), 
           ncol=6, frameon=True, fontsize=12)

plt.suptitle('Optimized Barnes-Hut Performance Scaling', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()